<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.10-agentic-rag/notebooks/exercises-6_10.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.10: Agentic RAG

*Module 6 · Retrieval-Augmented Generation (RAG)*

> Standard RAG always retrieves, even when it shouldn’t. Ask “What’s 2+2?” and it still searches your documents. Agentic RAG gives the LLM judgment : decide WHETHER to retrieve, WHICH source to search, and WHETHER the results are good enough. If not — retry with a better query. This is RAG with a brain.

`Query Routing` · `Adaptive Retrieval` · `Self-Correction` · `Tool-Based RAG` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.10.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Query Router — Direct Answer vs Retrieval vs Web Search — `01_query_router.py`
2. Step 2 — Self-Correcting Retrieval — Retry When Results Are Bad — `02_self_correcting.py`
3. Step 3 — RAG as a Tool — The Agent Chooses When to Use It — `03_rag_as_tool.py`
4. Step 4 — Multi-Source Router — Pick the Right Knowledge Base — `04_multi_source.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Query Router — Direct Answer vs Retrieval vs Web Search

The agent classifies each query: answer from knowledge, search docs, or search the web.

**`01_query_router.py`** · _Block 1 of 4_

QUERY ROUTER — Agent decides: answer directly, search docs, or search web


In [ ]:
# QUERY ROUTER — Agent decides: answer directly, search docs, or search web
import google.generativeai as genai, os, json
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class QueryRouter:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def route(self, query):
        resp = self.model.generate_content(
            f"Classify this query into ONE category. Return ONLY the category name.\n\n"
            f"Categories:\n"
            f"- DIRECT: general knowledge, math, greetings (no retrieval needed)\n"
            f"- DOCS: company-specific info, policies, pricing (search internal docs)\n"
            f"- WEB: current events, external facts, real-time data (search the web)\n\n"
            f"Query: {query}\nCategory:")
        category = resp.text.strip().upper().split()[0]
        if category not in ("DIRECT","DOCS","WEB"): category = "DOCS"
        return category

router = QueryRouter()
queries = [
    ("What is 2+2?",                        "DIRECT"),
    ("What is the refund policy?",           "DOCS"),
    ("Who won the IPL match yesterday?",     "WEB"),
    ("How much does the GenAI course cost?", "DOCS"),
    ("Hello! How are you?",                 "DIRECT"),
    ("What is the latest Python version?",   "WEB"),
]

print("Query Routing:\n")
correct = 0
for q, expected in queries:
    route = router.route(q)
    match = "✓" if route == expected else "✗"
    if route == expected: correct += 1
    print(f"  {match} '{q[:40]}' → {route} (expected: {expected})")
print(f"\n  Accuracy: {correct}/{len(queries)}")
print(f"  DIRECT = skip retrieval (save tokens + latency)")
print(f"  DOCS = search vector store")
print(f"  WEB = search the internet")


> **What just happened?** “What is 2+2?” routed to DIRECT (no retrieval). “Refund policy?” routed to DOCS (search vector store). “IPL match?” routed to WEB. Without routing, all 6 queries would search the vector store — wasting tokens on greetings and math. Routing saves 30-50% of retrieval calls in production.


### Step 2 · Self-Correcting Retrieval — Retry When Results Are Bad

The agent checks retrieval quality. If results are irrelevant, it rephrases and tries again.

**`02_self_correcting.py`** · _Block 2 of 4_

SELF-CORRECTING RETRIEVAL — Check quality, retry if bad


In [ ]:
# SELF-CORRECTING RETRIEVAL — Check quality, retry if bad
import google.generativeai as genai, numpy as np, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class SelfCorrectingRAG:
    def __init__(self, docs, max_retries=2):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.max_retries = max_retries
        self.chunks, self.embs = [], []
        for d in docs:
            e = genai.embed_content(model="models/text-embedding-004",content=d["text"])["embedding"]
            self.chunks.append(d); self.embs.append(np.array(e))

    def _retrieve(self, query, k=3):
        qe = np.array(genai.embed_content(model="models/text-embedding-004",content=query)["embedding"])
        scores = sorted([(i,np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e)))
                         for i,e in enumerate(self.embs)], key=lambda x:-x[1])[:k]
        return [(self.chunks[i], s) for i,s in scores]

    def _is_relevant(self, query, docs):
        """LLM judges if retrieved docs are relevant enough."""
        context = "\n".join(d["text"][:100] for d,_ in docs)
        resp = self.model.generate_content(
            f"Are these documents relevant to the query? Answer YES or NO only.\n"
            f"Query: {query}\nDocuments: {context}")
        return "YES" in resp.text.upper()

    def _rephrase(self, query, attempt):
        resp = self.model.generate_content(
            f"Rephrase this query differently to find better search results.\n"
            f"Original: {query}\nAttempt {attempt+1}. Return ONLY the rephrased query.")
        return resp.text.strip()

    def query(self, question):
        current_query = question
        for attempt in range(self.max_retries + 1):
            docs = self._retrieve(current_query)

            if self._is_relevant(question, docs):
                context = "\n".join(d["text"] for d,_ in docs)
                resp = self.model.generate_content(
                    f"Answer from context ONLY.\nContext:\n{context}\nQuestion: {question}")
                return {"answer":resp.text.strip(), "attempts":attempt+1,
                        "final_query":current_query, "status":"found"}

            # Self-correct: rephrase and retry
            if attempt < self.max_retries:
                current_query = self._rephrase(question, attempt)

        return {"answer":"I couldn't find relevant information after multiple attempts.",
                "attempts":self.max_retries+1, "status":"exhausted"}

# Demo
docs = [
    {"text":"Netsetos refund: full within 7 days, 50% 8-30 days, none after 30."},
    {"text":"GenAI course: 14999 rupees. 146 hours. Python + GCP."},
    {"text":"Prerequisites: basic Python, high school math. No ML needed."},
    {"text":"EMI available via Razorpay. Group discount: 20% off for 3+ students."},
]
rag = SelfCorrectingRAG(docs, max_retries=2)

print("Self-Correcting RAG:\n")
for q in ["Can I pay in installments?", "What's the return policy?", "Do you teach blockchain?"]:
    r = rag.query(q)
    print(f"  Q: {q}")
    print(f"  A: {r['answer'][:100]}")
    print(f"  [{r['status']}] Attempts: {r['attempts']}, Final query: {r.get('final_query','')[:50]}\n")


> **What just happened?** “Can I pay in installments?” might not match “EMI via Razorpay” on the first try. The agent checked relevance, found it weak, rephrased to “payment plan options” or “EMI installment available”, and retried. Self-correction catches 15-25% of queries that naive retrieval misses. The LLM acts as a quality gate: retrieve → check → retry.


### Step 3 · RAG as a Tool — The Agent Chooses When to Use It

RAG becomes one of several tools the agent can call, alongside calculator, web search, etc.

**`03_rag_as_tool.py`** · _Block 3 of 4_

RAG AS A TOOL — Agent with multiple tools including RAG


In [ ]:
# RAG AS A TOOL — Agent with multiple tools including RAG
import google.generativeai as genai, numpy as np, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# ── Build a simple RAG index ──
docs = [
    "GenAI course: 14999 rupees. 146 hours, 14 modules.",
    "Refund: full within 7 days, 50% 8-30 days, none after 30.",
    "Live classes 7 PM IST daily. EMI via Razorpay.",
]
embs = [np.array(genai.embed_content(model="models/text-embedding-004",content=d)["embedding"]) for d in docs]

# ── Define tools for the agent ──
def search_docs(query: str) -> str:
    """Search Netsetos knowledge base for company-specific information."""
    qe = np.array(genai.embed_content(model="models/text-embedding-004",content=query)["embedding"])
    scores = sorted([(i,np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e)))
                     for i,e in enumerate(embs)], key=lambda x:-x[1])
    return " | ".join(docs[i] for i,_ in scores[:2])

def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try: return str(eval(expression, {"__builtins__":{}}))
    except: return "Error"

def get_current_time() -> str:
    """Get current date and time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M IST")

# ── Agent with RAG as one tool ──
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    tools=[search_docs, calculate, get_current_time],
    system_instruction="You are a Netsetos assistant. Use tools when needed. For company info use search_docs. For math use calculate. Answer concisely."
)
chat = model.start_chat(enable_automatic_function_calling=True)

print("RAG as Tool (Agent decides):\n")
for q in [
    "What is the refund policy?",           # → search_docs
    "How much is 14999 with 18% GST?",     # → calculate
    "What time is it?",                     # → get_current_time
    "What's the price and when are classes?", # → search_docs
    "Hi there!",                            # → direct (no tool)
]:
    resp = chat.send_message(q)
    print(f"  Q: {q}")
    print(f"  A: {resp.text.strip()[:120]}\n")


> **What just happened?** Refund question → agent called search_docs . GST calculation → agent called calculate . Time → get_current_time . “Hi there!” → no tool, direct response. RAG is now just ONE of the agent’s tools. The LLM decides when to search documents, when to compute, and when to just respond. This is the architecture behind ChatGPT, Gemini Chat, and every production AI assistant.


### Step 4 · Multi-Source Router — Pick the Right Knowledge Base

Route to different vector stores based on query topic: FAQ, technical docs, billing, etc.

**`04_multi_source.py`** · _Block 4 of 4_

MULTI-SOURCE RAG — Route queries to the right knowledge base


In [ ]:
# MULTI-SOURCE RAG — Route queries to the right knowledge base
import google.generativeai as genai, numpy as np, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class MultiSourceRAG:
    """Route queries to the best knowledge base."""
    def __init__(self):
        self.sources = {}
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def add_source(self, name, description, docs):
        embs = [np.array(genai.embed_content(model="models/text-embedding-004",content=d)["embedding"]) for d in docs]
        self.sources[name] = {"desc":description, "docs":docs, "embs":embs}

    def _route(self, query):
        source_list = "\n".join(f"- {n}: {s['desc']}" for n,s in self.sources.items())
        resp = self.model.generate_content(
            f"Which source best answers this query? Return ONLY the source name.\n\n"
            f"Sources:\n{source_list}\n\nQuery: {query}\nBest source:")
        name = resp.text.strip().lower()
        for k in self.sources:
            if k.lower() in name: return k
        return list(self.sources.keys())[0]

    def query(self, question):
        source = self._route(question)
        s = self.sources[source]
        qe = np.array(genai.embed_content(model="models/text-embedding-004",content=question)["embedding"])
        scores = sorted([(i,np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e)))
                         for i,e in enumerate(s["embs"])], key=lambda x:-x[1])[:2]
        ctx = "\n".join(s["docs"][i] for i,_ in scores)
        ans = self.model.generate_content(f"Answer from context.\nContext:\n{ctx}\nQ: {question}")
        return {"answer":ans.text.strip(), "source":source}

rag = MultiSourceRAG()
rag.add_source("faq", "General questions, courses, schedules",
    ["Live classes 7PM IST daily. Recordings in 2 hours.",
     "GenAI course: 14 modules, 146 hours, Python + GCP."])
rag.add_source("billing", "Pricing, payments, refunds, EMI",
    ["GenAI: 14999 rupees. EMI via Razorpay.",
     "Refund: full 7 days, 50% 8-30 days, none after 30."])
rag.add_source("technical", "Prerequisites, curriculum, tools",
    ["Prerequisites: Python basics, high school math.",
     "Tools used: Google Colab, Vertex AI, ChromaDB, LangChain."])

print("Multi-Source RAG:\n")
for q in ["Can I get a refund?", "What tools will I learn?", "When are classes?"]:
    r = rag.query(q)
    print(f"  Q: {q}")
    print(f"  Routed to: [{r['source']}] → {r['answer'][:80]}\n")


> **What just happened?** Five patterns, one maturity ladder: Start with CRAG (one evaluator layer, plug-and-play, 10.5% hallucination). Add Adaptive RAG routing to save cost on simple queries. Graduate to Self-RAG when you can fine-tune (5.8% hallucination — lowest). Use FLARE for long-form generation, IRCoT for multi-hop reasoning. CRAG's key innovation: knowledge refinement strips — decompose docs into fine-grained strips, score each individually, filter noise even from relevant docs.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
